# 🧠 SkillSyncAI — AI-Based Hackathon Team Formation

**College Project — MVP / Prototype**

This notebook implements an AI-powered pipeline that:
1. Parses student resumes (PDFs)
2. Extracts skills & roles using NLP
3. Builds student profiles with scoring
4. Parses project requirements
5. Calculates match scores
6. Forms balanced teams with fairness constraints
7. Produces explainable JSON output

---

## 📦 Step 1: Install & Import Dependencies

In [ ]:
# === Install required packages (run once) ===
# Uncomment the line below if running on Google Colab
# !pip install pdfplumber spacy pandas -q
# !python -m spacy download en_core_web_sm -q

In [ ]:
import os
import re
import json
import math
import warnings
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Optional

import pandas as pd
import pdfplumber
import spacy

warnings.filterwarnings('ignore')

# Load spaCy English model for NLP tasks
nlp = spacy.load("en_core_web_sm")

print("✅ All libraries loaded successfully.")

---
## 🗂 Step 2: Skill Dictionary & Role Taxonomy

All skills are mapped to one of **7 fixed roles**:  
`Frontend`, `Backend`, `Fullstack`, `ML/AI`, `Data`, `UI/UX`, `DevOps`

Synonyms (e.g., "Web Developer") are normalized into these roles.

In [ ]:
# ============================================================
# SKILL DICTIONARY — Maps keywords to role categories
# Each keyword is lowercase for case-insensitive matching
# ============================================================

SKILL_DICTIONARY = {
    "frontend": [
        "react", "reactjs", "react.js", "angular", "angularjs", "vue", "vuejs",
        "vue.js", "html", "html5", "css", "css3", "javascript", "typescript",
        "jquery", "bootstrap", "tailwind", "tailwindcss",  "sass", "scss",
        "webpack", "vite", "next.js", "nextjs", "nuxt", "svelte",
        "responsive design", "front-end", "front end", "redux", "zustand",
        "material ui", "chakra ui", "ant design", "styled components",
    ],
    "backend": [
        "node", "nodejs", "node.js", "express", "expressjs", "django",
        "flask", "fastapi", "spring", "spring boot", "springboot",
        "java", "python", "ruby", "rails", "ruby on rails", "php", "laravel",
        "golang", "go", "rust", "c#", ".net", "asp.net", "rest api", "restful",
        "graphql", "grpc", "microservices", "back-end", "back end",
        "mongodb", "mysql", "postgresql", "postgres", "sqlite", "redis",
        "firebase", "supabase", "dynamodb", "cassandra", "sql", "nosql",
        "orm", "sequelize", "prisma", "mongoose", "hibernate",
        "socket.io", "websocket", "authentication", "jwt", "oauth",
    ],
    "fullstack": [
        "fullstack", "full stack", "full-stack", "mern", "mean",
        "lamp", "mern stack", "mean stack", "t3 stack",
    ],
    "ml": [
        "machine learning", "deep learning", "tensorflow", "pytorch",
        "keras", "scikit-learn", "sklearn", "neural network", "neural networks",
        "nlp", "natural language processing", "computer vision", "opencv",
        "classification", "regression", "clustering", "random forest",
        "xgboost", "lightgbm", "gradient boosting", "svm",
        "support vector", "decision tree", "logistic regression",
        "recurrent neural", "convolutional neural", "cnn", "rnn", "lstm",
        "transformer", "bert", "gpt", "llm", "large language model",
        "hugging face", "huggingface", "generative ai", "gen ai",
        "reinforcement learning", "model training", "model deployment",
        "feature engineering", "hyperparameter", "cross validation",
        "artificial intelligence", "ai", "ml",
    ],
    "data": [
        "data analysis", "data analytics", "data science", "data engineering",
        "pandas", "numpy", "matplotlib", "seaborn", "plotly",
        "data visualization", "data mining", "etl", "data pipeline",
        "big data", "spark", "hadoop", "apache spark", "kafka",
        "tableau", "power bi", "powerbi", "excel", "statistics",
        "statistical analysis", "hypothesis testing", "a/b testing",
        "data wrangling", "data cleaning", "exploratory data analysis",
        "eda", "jupyter", "notebook", "r programming",
    ],
    "uiux": [
        "figma", "adobe xd", "sketch", "invision", "zeplin",
        "ui", "ux", "ui/ux", "ui ux", "user interface", "user experience",
        "wireframe", "wireframing", "prototype", "prototyping",
        "design thinking", "user research", "usability testing",
        "interaction design", "visual design", "graphic design",
        "adobe photoshop", "photoshop", "illustrator", "canva",
        "color theory", "typography", "responsive design",
        "information architecture", "heuristic evaluation",
    ],
    "devops": [
        "docker", "kubernetes", "k8s", "aws", "azure", "gcp",
        "google cloud", "amazon web services", "cloud computing",
        "ci/cd", "cicd", "jenkins", "github actions", "gitlab ci",
        "terraform", "ansible", "linux", "bash", "shell scripting",
        "nginx", "apache", "devops", "dev ops", "site reliability",
        "monitoring", "prometheus", "grafana", "elk stack",
        "infrastructure", "deployment", "serverless", "lambda",
        "ec2", "s3", "load balancer", "networking",
        "git", "version control",
    ],
}

# Role display names for output
ROLE_DISPLAY_NAMES = {
    "frontend": "Frontend",
    "backend": "Backend",
    "fullstack": "Fullstack",
    "ml": "ML/AI",
    "data": "Data",
    "uiux": "UI/UX",
    "devops": "DevOps",
}

# Synonym normalization for role titles found in resumes
ROLE_SYNONYMS = {
    "web developer": "frontend",
    "front end developer": "frontend",
    "frontend developer": "frontend",
    "react developer": "frontend",
    "angular developer": "frontend",
    "ui developer": "frontend",
    "backend developer": "backend",
    "back end developer": "backend",
    "server side developer": "backend",
    "api developer": "backend",
    "java developer": "backend",
    "python developer": "backend",
    "node developer": "backend",
    "software engineer": "fullstack",
    "software developer": "fullstack",
    "sde": "fullstack",
    "full stack developer": "fullstack",
    "fullstack developer": "fullstack",
    "mern stack developer": "fullstack",
    "mean stack developer": "fullstack",
    "ml engineer": "ml",
    "machine learning engineer": "ml",
    "ai engineer": "ml",
    "deep learning engineer": "ml",
    "nlp engineer": "ml",
    "data scientist": "data",
    "data analyst": "data",
    "data engineer": "data",
    "business analyst": "data",
    "ui ux designer": "uiux",
    "ui/ux designer": "uiux",
    "ux designer": "uiux",
    "ui designer": "uiux",
    "graphic designer": "uiux",
    "product designer": "uiux",
    "devops engineer": "devops",
    "cloud engineer": "devops",
    "site reliability engineer": "devops",
    "sre": "devops",
    "system administrator": "devops",
}

# Experience keywords for heuristic scoring
EXPERIENCE_KEYWORDS = {
    "internship": 1.0,
    "intern": 1.0,
    "hackathon": 0.8,
    "project": 0.5,
    "freelance": 0.8,
    "work experience": 1.2,
    "professional experience": 1.2,
    "research": 0.7,
    "publication": 0.9,
    "open source": 0.7,
    "contribution": 0.4,
    "certification": 0.6,
    "certified": 0.6,
    "award": 0.5,
    "achievement": 0.4,
    "competition": 0.6,
    "winner": 0.7,
    "runner up": 0.5,
    "lead": 0.8,
    "team lead": 1.0,
    "mentor": 0.6,
}

ALL_ROLES = list(SKILL_DICTIONARY.keys())
print(f"✅ Skill Dictionary loaded: {len(SKILL_DICTIONARY)} roles")
print(f"   Roles: {', '.join(ROLE_DISPLAY_NAMES.values())}")
total_keywords = sum(len(v) for v in SKILL_DICTIONARY.values())
print(f"   Total skill keywords: {total_keywords}")

---
## 📄 Step 3: Resume PDF Parsing

Upload your PDF resumes to a folder (e.g., `/content/resumes/` on Colab).  
The parser extracts raw text from each PDF.

In [ ]:
# ============================================================
# PDF TEXT EXTRACTION — Converts resume PDFs to plain text
# ============================================================

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract all text from a PDF file using pdfplumber.
    Falls back gracefully if a page fails to parse.
    """
    text = ""
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
    except Exception as e:
        print(f"   ⚠️ Error reading {pdf_path}: {e}")
    return text.strip()


def extract_name_from_filename(filename: str) -> str:
    """
    Extract student name from the PDF filename.
    
    Examples:
      'John_Doe.pdf' -> 'John Doe'
      'jane-smith-resume.pdf' -> 'Jane Smith'
      'RAHUL KUMAR.pdf' -> 'Rahul Kumar'
    """
    # Remove extension
    name = os.path.splitext(os.path.basename(filename))[0]
    
    # Remove common suffixes like 'resume', 'cv', 'Resume', etc.
    name = re.sub(r'[-_\s]*(resume|cv|Resume|CV|RESUME)[-_\s]*', ' ', name, flags=re.IGNORECASE)
    
    # Replace underscores and hyphens with spaces
    name = name.replace('_', ' ').replace('-', ' ')
    
    # Clean up multiple spaces
    name = re.sub(r'\s+', ' ', name).strip()
    
    # Title case the name (handles 'JOHN DOE' -> 'John Doe')
    name = name.title()
    
    return name if name else "Unknown"


def load_all_resumes(resume_folder: str) -> List[Dict]:
    """
    Load all PDF resumes from a folder.
    Returns a list of dicts with: student_id, filename, name, raw_text
    """
    resumes = []
    pdf_files = sorted([
        f for f in os.listdir(resume_folder)
        if f.lower().endswith('.pdf')
    ])

    print(f"📂 Found {len(pdf_files)} PDF files in '{resume_folder}'")
    print("   Extracting text...\n")

    for idx, filename in enumerate(pdf_files, 1):
        filepath = os.path.join(resume_folder, filename)
        raw_text = extract_text_from_pdf(filepath)

        if not raw_text:
            print(f"   ⚠️ [{idx}] {filename} — No text extracted (skipped)")
            continue

        name = extract_name_from_filename(filename)
        student_id = f"S{idx:03d}"

        resumes.append({
            "student_id": student_id,
            "filename": filename,
            "name": name,
            "raw_text": raw_text,
        })
        print(f"   ✅ [{idx}] {filename} → {name} ({len(raw_text)} chars)")

    print(f"\n📊 Successfully parsed {len(resumes)} / {len(pdf_files)} resumes.")
    return resumes

In [ ]:
# ============================================================
# 📁 CONFIGURE YOUR RESUME FOLDER PATH HERE
# ============================================================
# On Google Colab, upload your resumes to /content/resumes/
# Or mount Google Drive and point to the folder.
#
# Example paths:
#   Colab:  "/content/resumes"
#   Drive:  "/content/drive/MyDrive/hackathon_resumes"
#   Local:  "./resumes"

RESUME_FOLDER = "./resumes"  # <-- CHANGE THIS TO YOUR PATH

# Load all resumes
raw_resumes = load_all_resumes(RESUME_FOLDER)

---
## 🔍 Step 4: Skill Extraction Engine (NLP)

- Rule-based keyword matching against the skill dictionary  
- Counts keyword occurrences per role  
- Normalizes scores to 0–10 scale  
- Determines primary role  

In [ ]:
# ============================================================
# SKILL EXTRACTION — Rule-based NLP with keyword matching
# ============================================================

def preprocess_text(text: str) -> str:
    """
    Clean and normalize resume text for keyword matching.
    """
    text = text.lower()
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove special characters but keep periods, slashes, hyphens (for tech terms)
    text = re.sub(r'[^a-z0-9\s\.\-\/\#\+]', ' ', text)
    return text.strip()


def extract_skills(text: str) -> Dict[str, Dict]:
    """
    Extract skills from resume text using keyword matching.

    Returns:
        Dict with role -> {raw_count, score (0-10), matched_keywords}
    """
    cleaned = preprocess_text(text)
    role_data = {}

    for role, keywords in SKILL_DICTIONARY.items():
        matched = []
        raw_count = 0

        for keyword in keywords:
            # Use word boundary matching for short keywords to avoid false positives
            if len(keyword) <= 3:
                pattern = r'\b' + re.escape(keyword) + r'\b'
            else:
                pattern = re.escape(keyword)

            count = len(re.findall(pattern, cleaned))
            if count > 0:
                matched.append(keyword)
                raw_count += count

        role_data[role] = {
            "raw_count": raw_count,
            "matched_keywords": matched,
            "unique_skills": len(matched),
        }

    # Normalize scores to 0-10 scale
    # Using a combination of unique skills matched and frequency
    max_possible = max(
        (r["unique_skills"] * 2 + r["raw_count"] * 0.5) for r in role_data.values()
    ) if any(r["raw_count"] > 0 for r in role_data.values()) else 1

    for role in role_data:
        r = role_data[role]
        raw_score = r["unique_skills"] * 2 + r["raw_count"] * 0.5
        # Normalize to 0-10, with diminishing returns
        normalized = min(10, round((raw_score / max(max_possible, 1)) * 10, 1))
        role_data[role]["score"] = normalized

    return role_data


def determine_primary_role(skill_data: Dict[str, Dict]) -> str:
    """
    Determine the student's primary role based on highest skill score.
    Tie-breaking: prefer role with more unique skills matched.
    """
    best_role = "backend"  # default fallback
    best_score = -1
    best_unique = -1

    for role, data in skill_data.items():
        score = data["score"]
        unique = data["unique_skills"]

        if score > best_score or (score == best_score and unique > best_unique):
            best_score = score
            best_unique = unique
            best_role = role

    # Special case: if fullstack has a high score, check if frontend + backend
    # are both strong — that reinforces the fullstack classification
    if best_role == "fullstack":
        fe_score = skill_data.get("frontend", {}).get("score", 0)
        be_score = skill_data.get("backend", {}).get("score", 0)
        if fe_score > be_score and fe_score > skill_data["fullstack"]["score"]:
            best_role = "frontend"
        elif be_score > fe_score and be_score > skill_data["fullstack"]["score"]:
            best_role = "backend"

    return best_role


def infer_role_from_title(text: str) -> Optional[str]:
    """
    Check if the resume contains explicit role titles that map to our taxonomy.
    Returns the role if found, None otherwise.
    """
    cleaned = text.lower()
    for title, role in ROLE_SYNONYMS.items():
        if title in cleaned:
            return role
    return None


print("✅ Skill extraction engine ready.")

---
## 📈 Step 5: Experience Scoring (Heuristic)

Counts experience-related keywords and normalizes to a **0–5** scale.

In [ ]:
# ============================================================
# EXPERIENCE SCORING — Heuristic based on keyword frequency
# ============================================================

def calculate_experience_score(text: str) -> Dict:
    """
    Calculate experience score from resume text.

    Strategy:
      - Count occurrences of experience keywords
      - Each keyword has a weight (see EXPERIENCE_KEYWORDS)
      - Weighted sum is normalized to 0-5 scale

    Returns:
        Dict with score, matched_keywords, and raw_total
    """
    cleaned = preprocess_text(text)
    matched = []
    weighted_total = 0.0

    for keyword, weight in EXPERIENCE_KEYWORDS.items():
        count = cleaned.count(keyword)
        if count > 0:
            # Cap per-keyword contribution to avoid one keyword dominating
            capped_count = min(count, 5)
            weighted_total += capped_count * weight
            matched.append({
                "keyword": keyword,
                "count": count,
                "contribution": round(capped_count * weight, 2)
            })

    # Normalize to 0-5 scale (typical resumes score 3-15 raw)
    # Using a sigmoid-like curve for smooth normalization
    normalized = round(min(5.0, (weighted_total / (weighted_total + 5)) * 5 * 2), 2)

    return {
        "score": normalized,
        "raw_total": round(weighted_total, 2),
        "matched_keywords": matched,
    }


print("✅ Experience scoring engine ready.")

---
## 👤 Step 6: Student Profile Generator

Combines skill extraction + experience scoring into a complete profile for each student.

In [ ]:
# ============================================================
# STUDENT PROFILE BUILDER — Combines all extraction results
# ============================================================

def build_student_profile(resume_data: Dict) -> Dict:
    """
    Build a complete student profile from raw resume data.

    Input: dict with student_id, name, filename, raw_text
    Output: complete profile with skills, role, experience, etc.
    """
    text = resume_data["raw_text"]

    # 1. Extract skills
    skill_data = extract_skills(text)

    # 2. Determine primary role
    primary_role = determine_primary_role(skill_data)

    # 3. Check for explicit role title in resume (gives bonus confidence)
    title_role = infer_role_from_title(text)
    if title_role and title_role != primary_role:
        # If explicit title differs, check if its score is close to primary
        title_score = skill_data.get(title_role, {}).get("score", 0)
        primary_score = skill_data.get(primary_role, {}).get("score", 0)
        # Prefer explicit title if it's within 30% of primary role score
        if title_score >= primary_score * 0.7:
            primary_role = title_role

    # 4. Calculate experience score
    experience = calculate_experience_score(text)

    # 5. Calculate skill diversity (how many roles have non-zero scores)
    skill_diversity = sum(
        1 for r in skill_data.values() if r["score"] > 0
    ) / len(ALL_ROLES)

    # 6. Collect top matched skills for explainability
    top_skills = []
    for role in ALL_ROLES:
        for kw in skill_data[role]["matched_keywords"][:3]:  # Top 3 per role
            top_skills.append(kw)

    profile = {
        "student_id": resume_data["student_id"],
        "name": resume_data["name"],
        "filename": resume_data["filename"],
        "skills": {role: data["score"] for role, data in skill_data.items()},
        "skill_details": skill_data,  # Full details for explainability
        "primary_role": primary_role,
        "experience_score": experience["score"],
        "experience_details": experience,
        "skill_diversity": round(skill_diversity, 2),
        "top_skills": top_skills,
    }

    return profile


def build_all_profiles(resumes: List[Dict]) -> List[Dict]:
    """
    Build profiles for all parsed resumes.
    """
    print("🧠 Building student profiles...\n")
    profiles = []

    for resume in resumes:
        profile = build_student_profile(resume)
        profiles.append(profile)
        role_display = ROLE_DISPLAY_NAMES.get(profile['primary_role'], profile['primary_role'])
        print(
            f"   ✅ {profile['student_id']} | {profile['name']:<25} | "
            f"Role: {role_display:<10} | "
            f"Exp: {profile['experience_score']:.1f}/5 | "
            f"Diversity: {profile['skill_diversity']:.2f}"
        )

    print(f"\n📊 Built {len(profiles)} student profiles.")

    # Summary statistics
    role_counts = Counter(p["primary_role"] for p in profiles)
    print("\n📈 Role Distribution:")
    for role, count in role_counts.most_common():
        bar = "█" * count
        print(f"   {ROLE_DISPLAY_NAMES.get(role, role):<12} {count:>3}  {bar}")

    return profiles


print("✅ Profile builder ready.")

In [ ]:
# ============================================================
# BUILD ALL STUDENT PROFILES
# ============================================================

student_profiles = build_all_profiles(raw_resumes)

In [ ]:
# ============================================================
# PREVIEW: View detailed profile of the first student
# ============================================================

if student_profiles:
    sample = student_profiles[0]
    print(f"\n🔍 Sample Profile: {sample['name']}")
    print(f"   Student ID:    {sample['student_id']}")
    print(f"   Primary Role:  {ROLE_DISPLAY_NAMES.get(sample['primary_role'])}")
    print(f"   Experience:    {sample['experience_score']}/5")
    print(f"   Skill Diversity: {sample['skill_diversity']}")
    print(f"\n   Skill Scores:")
    for role, score in sample['skills'].items():
        bar = "█" * int(score)
        print(f"      {ROLE_DISPLAY_NAMES.get(role, role):<12} {score:>5.1f}  {bar}")
    print(f"\n   Top Keywords: {', '.join(sample['top_skills'][:10])}")

---
## 🎯 Step 7: Project Requirement Parsing

Parses plain-English project descriptions to extract:  
- Required roles  
- Desired team size  
- Priority skills

In [ ]:
# ============================================================
# PROJECT REQUIREMENT PARSER
# ============================================================

def parse_project_requirements(requirement_text: str, team_size: int = 4) -> Dict:
    """
    Parse a plain-English project requirement to extract needed roles.

    Args:
        requirement_text: Natural language description of the project
        team_size: Desired team size (default: 4)

    Returns:
        Dict with roles_needed, priority_skills, team_size, parsed_details
    """
    cleaned = preprocess_text(requirement_text)
    role_relevance = {}

    # Score each role based on keyword matches in the requirement
    for role, keywords in SKILL_DICTIONARY.items():
        matched = []
        for keyword in keywords:
            if keyword in cleaned:
                matched.append(keyword)

        if matched:
            role_relevance[role] = {
                "match_count": len(matched),
                "matched_keywords": matched,
            }

    # Also check for role synonyms directly mentioned
    for title, role in ROLE_SYNONYMS.items():
        if title in cleaned:
            if role not in role_relevance:
                role_relevance[role] = {"match_count": 0, "matched_keywords": []}
            role_relevance[role]["match_count"] += 2  # Bonus for explicit mention
            role_relevance[role]["matched_keywords"].append(f"[role: {title}]")

    # Sort roles by relevance
    sorted_roles = sorted(
        role_relevance.items(),
        key=lambda x: x[1]["match_count"],
        reverse=True
    )

    # Select top roles needed (up to team_size)
    roles_needed = [role for role, _ in sorted_roles[:team_size]]

    # If fewer roles detected than team_size, add generic roles
    if len(roles_needed) < team_size:
        # Fill with common roles not already selected
        default_fill = ["backend", "frontend", "fullstack", "data", "devops", "ml", "uiux"]
        for fill_role in default_fill:
            if fill_role not in roles_needed and len(roles_needed) < team_size:
                roles_needed.append(fill_role)

    # Collect priority skills (specific technologies mentioned)
    priority_skills = []
    for role_info in role_relevance.values():
        for kw in role_info["matched_keywords"]:
            if not kw.startswith("[role:"):
                priority_skills.append(kw)

    result = {
        "requirement_text": requirement_text,
        "roles_needed": roles_needed,
        "team_size": team_size,
        "priority_skills": priority_skills,
        "role_relevance": {
            role: data for role, data in sorted_roles
        },
    }

    return result


print("✅ Project requirement parser ready.")

In [ ]:
# ============================================================
# 📝 DEFINE YOUR PROJECT REQUIREMENTS HERE
# ============================================================
# Add one or more projects. Each project has:
#   - name: Project title
#   - description: Plain English requirements
#   - team_size: Number of team members needed

PROJECTS = [
    {
        "name": "Smart Campus Assistant",
        "description": "We need a React frontend, Node.js backend with REST APIs, and an ML model for predicting campus facility usage.",
        "team_size": 4,
    },
    {
        "name": "Health Data Dashboard",
        "description": "Build a data visualization dashboard using Python, pandas, and Plotly. Backend with Flask and PostgreSQL. UI/UX design needed.",
        "team_size": 4,
    },
    {
        "name": "E-Commerce Platform",
        "description": "Full-stack e-commerce with React frontend, Express.js backend, MongoDB database, Docker deployment, and payment integration.",
        "team_size": 5,
    },
    # Add more projects as needed:
    # {
    #     "name": "Your Project Name",
    #     "description": "Your project description...",
    #     "team_size": 4,
    # },
]

# Parse all project requirements
parsed_projects = []
for project in PROJECTS:
    parsed = parse_project_requirements(
        project["description"],
        team_size=project["team_size"]
    )
    parsed["project_name"] = project["name"]
    parsed_projects.append(parsed)

    print(f"\n🎯 Project: {project['name']}")
    print(f"   Team Size:  {parsed['team_size']}")
    roles_display = [ROLE_DISPLAY_NAMES.get(r, r) for r in parsed['roles_needed']]
    print(f"   Roles:      {', '.join(roles_display)}")
    print(f"   Key Skills: {', '.join(parsed['priority_skills'][:8])}")

print(f"\n📊 Total projects: {len(parsed_projects)}")
total_slots = sum(p['team_size'] for p in parsed_projects)
print(f"   Total team slots: {total_slots}")
print(f"   Available students: {len(student_profiles)}")

---
## 📊 Step 8: Matching Score Calculation

**Formula:**  
$$\text{match\_score} = 0.6 \times \text{role\_skill\_score} + 0.3 \times \text{experience\_score} + 0.1 \times \text{skill\_diversity}$$

Each student gets a match score **per project per role**.

In [ ]:
# ============================================================
# MATCHING SCORE CALCULATOR
# ============================================================

# Configurable weights (must sum to 1.0)
WEIGHT_SKILL = 0.6
WEIGHT_EXPERIENCE = 0.3
WEIGHT_DIVERSITY = 0.1


def calculate_match_score(
    student: Dict,
    role_needed: str,
    priority_skills: List[str] = None
) -> Dict:
    """
    Calculate how well a student matches a specific role for a project.

    Args:
        student: Student profile dict
        role_needed: The role to match against
        priority_skills: Optional list of priority tech skills

    Returns:
        Dict with total_score and component breakdown
    """
    # Component 1: Role skill score (0-10, normalized to 0-1)
    role_skill_score = student["skills"].get(role_needed, 0) / 10.0

    # Bonus if student's primary role matches the needed role
    primary_role_bonus = 0.15 if student["primary_role"] == role_needed else 0.0

    # Component 2: Experience score (0-5, normalized to 0-1)
    experience_normalized = student["experience_score"] / 5.0

    # Component 3: Skill diversity (already 0-1)
    diversity = student["skill_diversity"]

    # Priority skill bonus (small boost for matching specific technologies)
    priority_bonus = 0.0
    if priority_skills:
        student_keywords = set()
        for role_data in student["skill_details"].values():
            student_keywords.update(role_data["matched_keywords"])
        matching_priority = set(priority_skills) & student_keywords
        if matching_priority:
            priority_bonus = min(0.1, len(matching_priority) * 0.02)

    # Final weighted score
    total_score = (
        WEIGHT_SKILL * (role_skill_score + primary_role_bonus) +
        WEIGHT_EXPERIENCE * experience_normalized +
        WEIGHT_DIVERSITY * diversity +
        priority_bonus
    )

    # Clamp to 0-1
    total_score = round(min(1.0, max(0.0, total_score)), 4)

    return {
        "total_score": total_score,
        "components": {
            "role_skill": round(role_skill_score, 3),
            "primary_role_bonus": round(primary_role_bonus, 3),
            "experience": round(experience_normalized, 3),
            "diversity": round(diversity, 3),
            "priority_bonus": round(priority_bonus, 3),
        }
    }


def calculate_all_match_scores(
    profiles: List[Dict],
    projects: List[Dict]
) -> Dict:
    """
    Calculate match scores for all student-project-role combinations.

    Returns:
        Nested dict: project_name -> role -> [(student_id, score_data)]
        sorted by score descending.
    """
    all_scores = {}

    for project in projects:
        project_name = project["project_name"]
        all_scores[project_name] = {}

        for role in project["roles_needed"]:
            role_scores = []

            for student in profiles:
                score_data = calculate_match_score(
                    student, role, project.get("priority_skills", [])
                )
                role_scores.append((
                    student["student_id"],
                    student["name"],
                    score_data
                ))

            # Sort by total score descending
            role_scores.sort(key=lambda x: x[2]["total_score"], reverse=True)
            all_scores[project_name][role] = role_scores

    return all_scores


print("✅ Matching score calculator ready.")

In [ ]:
# ============================================================
# CALCULATE ALL MATCH SCORES
# ============================================================

match_scores = calculate_all_match_scores(student_profiles, parsed_projects)

# Preview top 3 candidates per role for first project
if parsed_projects:
    first_project = parsed_projects[0]
    print(f"\n🔍 Top candidates for: {first_project['project_name']}")
    print("=" * 70)

    for role in first_project["roles_needed"]:
        role_display = ROLE_DISPLAY_NAMES.get(role, role)
        print(f"\n   📌 {role_display}:")
        candidates = match_scores[first_project["project_name"]][role][:5]
        for rank, (sid, name, score_data) in enumerate(candidates, 1):
            score = score_data['total_score']
            bar = "█" * int(score * 20)
            print(f"      {rank}. {name:<22} {score:.3f}  {bar}")

---
## ⚖️ Step 9: Balanced Team Formation (Core Intelligence)

**Algorithm: Greedy + Round-Robin + Fairness**

Constraints:  
- One student per role (if possible)  
- No student on multiple teams  
- Avoid top-heavy teams (spread talent evenly)  
- Fair distribution across all teams

In [ ]:
# ============================================================
# TEAM FORMATION ENGINE — Greedy + Fairness Balancing
# ============================================================

def form_balanced_teams(
    profiles: List[Dict],
    projects: List[Dict],
    match_scores: Dict,
    fairness_mode: str = "round_robin"
) -> List[Dict]:
    """
    Form balanced teams for all projects.

    Algorithm:
    1. Group students by primary role
    2. For each project, use round-robin draft to pick candidates
    3. Within each round, alternate between picking best and mid-tier
    4. Enforce: no student assigned to multiple teams

    Args:
        profiles: List of student profiles
        projects: List of parsed project requirements
        match_scores: Pre-calculated match scores
        fairness_mode: 'greedy' (best first) or 'round_robin' (balanced)

    Returns:
        List of team dicts with members, scores, and balance info
    """
    assigned_students = set()  # Track which students are already assigned
    teams = []

    # Build lookup: student_id -> profile
    profile_lookup = {p["student_id"]: p for p in profiles}

    # Sort projects by team_size descending (larger teams get priority)
    sorted_projects = sorted(projects, key=lambda p: p["team_size"], reverse=True)

    if fairness_mode == "round_robin":
        # Round-robin: alternate which project picks first in each round
        # This prevents the first project from always getting the best candidates
        teams = _round_robin_assignment(
            sorted_projects, match_scores, profile_lookup, assigned_students
        )
    else:
        # Greedy: each project picks best available in order
        teams = _greedy_assignment(
            sorted_projects, match_scores, profile_lookup, assigned_students
        )

    # Calculate team balance scores
    for team in teams:
        team["balance_score"] = _calculate_team_balance(team)

    # Log unassigned students
    all_ids = {p["student_id"] for p in profiles}
    unassigned = all_ids - assigned_students
    if unassigned:
        print(f"\n⚠️ {len(unassigned)} students were not assigned to any team.")
        for sid in sorted(unassigned):
            name = profile_lookup[sid]["name"]
            role = profile_lookup[sid]["primary_role"]
            print(f"   - {sid}: {name} ({ROLE_DISPLAY_NAMES.get(role, role)})")

    return teams


def _round_robin_assignment(projects, match_scores, profile_lookup, assigned):
    """
    Round-robin team formation for fairness.
    Each project fills one role slot per round.
    Projects alternate who picks first each round.
    """
    teams = []
    project_teams = {}  # project_name -> {members: [], roles_filled: set()}

    for project in projects:
        project_teams[project["project_name"]] = {
            "project_name": project["project_name"],
            "required_roles": project["roles_needed"][:],
            "team_size": project["team_size"],
            "members": [],
            "roles_filled": set(),
            "unfilled_roles": [],
        }

    max_team_size = max(p["team_size"] for p in projects)

    for round_num in range(max_team_size):
        # Alternate order each round for fairness
        if round_num % 2 == 0:
            round_order = projects[:]
        else:
            round_order = projects[::-1]

        for project in round_order:
            pname = project["project_name"]
            team = project_teams[pname]

            # Skip if team is already full
            if len(team["members"]) >= team["team_size"]:
                continue

            # Determine which role to fill this round
            unfilled = [
                r for r in team["required_roles"]
                if r not in team["roles_filled"]
            ]

            if not unfilled:
                # All required roles filled but team not full
                # Fill remaining with best available student for any role
                best_candidate = _find_best_available(
                    None, pname, match_scores, profile_lookup, assigned
                )
                if best_candidate:
                    team["members"].append(best_candidate)
                    assigned.add(best_candidate["student_id"])
                continue

            role_to_fill = unfilled[0]

            # Use round-robin tier selection:
            # Even rounds: pick from top tier
            # Odd rounds: pick from mid tier (for balance)
            tier = "top" if round_num % 2 == 0 else "mid"

            candidate = _find_best_available(
                role_to_fill, pname, match_scores, profile_lookup, assigned, tier
            )

            if candidate:
                team["members"].append(candidate)
                team["roles_filled"].add(role_to_fill)
                assigned.add(candidate["student_id"])
            else:
                team["unfilled_roles"].append(role_to_fill)

    # Convert to final team format
    for pname, team in project_teams.items():
        teams.append(team)

    return teams


def _greedy_assignment(projects, match_scores, profile_lookup, assigned):
    """Simple greedy: each project picks best available per role."""
    teams = []

    for project in projects:
        team = {
            "project_name": project["project_name"],
            "required_roles": project["roles_needed"],
            "team_size": project["team_size"],
            "members": [],
            "roles_filled": set(),
            "unfilled_roles": [],
        }

        for role in project["roles_needed"]:
            candidate = _find_best_available(
                role, project["project_name"],
                match_scores, profile_lookup, assigned, "top"
            )
            if candidate:
                team["members"].append(candidate)
                team["roles_filled"].add(role)
                assigned.add(candidate["student_id"])
            else:
                team["unfilled_roles"].append(role)

        # Fill remaining slots if team not full
        while len(team["members"]) < team["team_size"]:
            candidate = _find_best_available(
                None, project["project_name"],
                match_scores, profile_lookup, assigned, "top"
            )
            if candidate:
                team["members"].append(candidate)
                assigned.add(candidate["student_id"])
            else:
                break

        teams.append(team)

    return teams


def _find_best_available(
    role: Optional[str],
    project_name: str,
    match_scores: Dict,
    profile_lookup: Dict,
    assigned: set,
    tier: str = "top"
) -> Optional[Dict]:
    """
    Find the best available (unassigned) student for a role.

    tier='top': pick the highest-scoring unassigned student
    tier='mid': pick from the middle third for balance
    """
    if role and role in match_scores.get(project_name, {}):
        candidates = match_scores[project_name][role]
    else:
        # No specific role: get candidates from all roles and merge
        all_candidates = {}
        for role_key, cands in match_scores.get(project_name, {}).items():
            for sid, name, score_data in cands:
                if sid not in all_candidates or score_data["total_score"] > all_candidates[sid][2]["total_score"]:
                    all_candidates[sid] = (sid, name, score_data)
        candidates = sorted(all_candidates.values(), key=lambda x: x[2]["total_score"], reverse=True)

    if not candidates:
        return None

    # Filter out already assigned
    available = [(sid, name, sd) for sid, name, sd in candidates if sid not in assigned]

    if not available:
        return None

    # Select based on tier
    if tier == "mid" and len(available) >= 3:
        # Pick from the middle third
        start = len(available) // 3
        end = 2 * len(available) // 3
        selected = available[start]  # Best of middle tier
    else:
        selected = available[0]  # Top candidate

    sid, name, score_data = selected
    profile = profile_lookup[sid]
    assigned_role = role if role else profile["primary_role"]

    return {
        "student_id": sid,
        "name": name,
        "assigned_role": assigned_role,
        "primary_role": profile["primary_role"],
        "match_score": score_data["total_score"],
        "score_breakdown": score_data["components"],
        "experience_score": profile["experience_score"],
        "top_skills": profile["top_skills"][:5],
        "skill_scores": profile["skills"],
    }


def _calculate_team_balance(team: Dict) -> Dict:
    """
    Calculate balance metrics for a team.

    Returns dict with:
      - average_score: Mean match score of team
      - score_variance: Variance in match scores (lower = more balanced)
      - role_coverage: Fraction of required roles filled
      - overall_balance: Combined balance metric (0-1, higher = better)
    """
    if not team["members"]:
        return {"average_score": 0, "score_variance": 0, "role_coverage": 0, "overall_balance": 0}

    scores = [m["match_score"] for m in team["members"]]
    avg = sum(scores) / len(scores)
    variance = sum((s - avg) ** 2 for s in scores) / len(scores)

    required = len(team.get("required_roles", []))
    filled = len(team.get("roles_filled", set()))
    role_coverage = filled / max(required, 1)

    # Overall balance: high avg, low variance, high coverage
    balance = (
        0.4 * avg +
        0.3 * (1 - min(variance, 1)) +  # Penalize variance
        0.3 * role_coverage
    )

    return {
        "average_score": round(avg, 3),
        "score_variance": round(variance, 4),
        "role_coverage": round(role_coverage, 2),
        "overall_balance": round(balance, 3),
    }


print("✅ Team formation engine ready.")

In [ ]:
# ============================================================
# 🏗️ FORM TEAMS
# ============================================================
# Choose fairness_mode:
#   'round_robin' — balanced, ensures fair distribution (recommended)
#   'greedy'      — each project gets the best available in order

formed_teams = form_balanced_teams(
    profiles=student_profiles,
    projects=parsed_projects,
    match_scores=match_scores,
    fairness_mode="round_robin"  # Change to "greedy" if preferred
)

# Display formed teams
print("\n" + "=" * 70)
print("🏆 FORMED TEAMS")
print("=" * 70)

for team in formed_teams:
    balance = team['balance_score']
    print(f"\n🎯 {team['project_name']}")
    print(f"   Balance: {balance['overall_balance']:.2f} | "
          f"Avg Score: {balance['average_score']:.3f} | "
          f"Role Coverage: {balance['role_coverage']:.0%}")
    print(f"   {'─' * 60}")

    for member in team['members']:
        role_display = ROLE_DISPLAY_NAMES.get(member['assigned_role'], member['assigned_role'])
        print(
            f"   {member['student_id']} | {member['name']:<22} | "
            f"Role: {role_display:<10} | "
            f"Score: {member['match_score']:.3f} | "
            f"Exp: {member['experience_score']:.1f}"
        )

    if team.get('unfilled_roles'):
        unfilled = [ROLE_DISPLAY_NAMES.get(r, r) for r in team['unfilled_roles']]
        print(f"   ⚠️ Unfilled roles: {', '.join(unfilled)}")

---
## 🧾 Step 10: Explainable Output Generation

Every team member gets a **human-readable reason** explaining why they were selected.

In [ ]:
# ============================================================
# EXPLAINABLE AI — Generate reasons for each team assignment
# ============================================================

def generate_explanation(member: Dict, profile_lookup: Dict) -> str:
    """
    Generate a human-readable explanation for why a student was
    assigned to a specific role in a team.

    This is the Explainable AI (XAI) component.
    """
    reasons = []
    profile = profile_lookup.get(member["student_id"], {})
    assigned_role = member["assigned_role"]
    role_display = ROLE_DISPLAY_NAMES.get(assigned_role, assigned_role)

    # 1. Skill match reason
    skill_score = member["skill_scores"].get(assigned_role, 0)
    if skill_score >= 7:
        reasons.append(f"Strong {role_display} skills (score: {skill_score}/10)")
    elif skill_score >= 4:
        reasons.append(f"Moderate {role_display} skills (score: {skill_score}/10)")
    elif skill_score > 0:
        reasons.append(f"Some {role_display} experience (score: {skill_score}/10)")

    # 2. Primary role match
    if member["primary_role"] == assigned_role:
        reasons.append(f"Primary expertise aligns with {role_display} role")

    # 3. Relevant skills detail
    if profile and "skill_details" in profile:
        role_details = profile["skill_details"].get(assigned_role, {})
        matched = role_details.get("matched_keywords", [])
        if matched:
            top_kw = matched[:4]
            reasons.append(f"Key skills: {', '.join(top_kw)}")

    # 4. Experience reason
    exp = member["experience_score"]
    if exp >= 3.5:
        reasons.append(f"Strong practical experience (score: {exp}/5)")
    elif exp >= 2:
        reasons.append(f"Moderate experience with projects/internships (score: {exp}/5)")

    # 5. Experience details
    if profile and "experience_details" in profile:
        exp_keywords = profile["experience_details"].get("matched_keywords", [])
        exp_highlights = [
            ek["keyword"] for ek in exp_keywords
            if ek["contribution"] >= 0.5
        ][:3]
        if exp_highlights:
            reasons.append(f"Experience includes: {', '.join(exp_highlights)}")

    # 6. Match score context
    score = member["match_score"]
    if score >= 0.7:
        reasons.append(f"Top-tier match for this project (score: {score:.2f})")
    elif score >= 0.4:
        reasons.append(f"Good overall match (score: {score:.2f})")

    if not reasons:
        reasons.append("Selected to fill remaining team slot based on overall profile")

    return "; ".join(reasons)


def generate_all_explanations(teams: List[Dict], profiles: List[Dict]) -> List[Dict]:
    """
    Add explanations to all team members and generate final output.
    """
    profile_lookup = {p["student_id"]: p for p in profiles}
    explained_teams = []

    for team in teams:
        explained_team = {
            "project_name": team["project_name"],
            "team_size": team["team_size"],
            "balance_score": team["balance_score"],
            "members": [],
            "unfilled_roles": [
                ROLE_DISPLAY_NAMES.get(r, r)
                for r in team.get("unfilled_roles", [])
            ],
        }

        for member in team["members"]:
            explanation = generate_explanation(member, profile_lookup)

            explained_member = {
                "student_id": member["student_id"],
                "name": member["name"],
                "role": ROLE_DISPLAY_NAMES.get(member["assigned_role"], member["assigned_role"]),
                "match_score": member["match_score"],
                "experience_score": member["experience_score"],
                "reason": explanation,
                "score_breakdown": member["score_breakdown"],
            }
            explained_team["members"].append(explained_member)

        explained_teams.append(explained_team)

    return explained_teams


print("✅ Explanation generator ready.")

In [ ]:
# ============================================================
# GENERATE FINAL EXPLAINABLE OUTPUT
# ============================================================

final_teams = generate_all_explanations(formed_teams, student_profiles)

# Pretty print the results
print("\n" + "=" * 70)
print("📋 FINAL TEAM ASSIGNMENTS WITH EXPLANATIONS")
print("=" * 70)

for team in final_teams:
    balance = team['balance_score']
    print(f"\n{'━' * 70}")
    print(f"🎯 PROJECT: {team['project_name']}")
    print(f"   Team Balance: {balance['overall_balance']:.2f}/1.00 | "
          f"Role Coverage: {balance['role_coverage']:.0%}")
    print(f"{'━' * 70}")

    for i, member in enumerate(team['members'], 1):
        print(f"\n   👤 Member {i}: {member['name']}")
        print(f"      ID:     {member['student_id']}")
        print(f"      Role:   {member['role']}")
        print(f"      Score:  {member['match_score']:.3f}")
        print(f"      Reason: {member['reason']}")

    if team['unfilled_roles']:
        print(f"\n   ⚠️ Unfilled: {', '.join(team['unfilled_roles'])}")

---
## 💾 Step 11: Export Results (JSON & CSV)

Export final teams, student profiles, and match data for the frontend team.

In [ ]:
# ============================================================
# EXPORT RESULTS — JSON & CSV
# ============================================================

OUTPUT_DIR = "./output"  # Change this path as needed
os.makedirs(OUTPUT_DIR, exist_ok=True)


# === 1. Export Final Teams (JSON) ===
teams_output_path = os.path.join(OUTPUT_DIR, "final_teams.json")
with open(teams_output_path, "w", encoding="utf-8") as f:
    json.dump(final_teams, f, indent=2, ensure_ascii=False)
print(f"✅ Teams exported to: {teams_output_path}")


# === 2. Export Student Profiles (JSON) ===
# Clean profiles for export (remove raw skill_details to reduce size)
profiles_for_export = []
for p in student_profiles:
    export_profile = {
        "student_id": p["student_id"],
        "name": p["name"],
        "primary_role": ROLE_DISPLAY_NAMES.get(p["primary_role"], p["primary_role"]),
        "skills": p["skills"],
        "experience_score": p["experience_score"],
        "skill_diversity": p["skill_diversity"],
        "top_skills": p["top_skills"][:8],
    }
    profiles_for_export.append(export_profile)

profiles_output_path = os.path.join(OUTPUT_DIR, "student_profiles.json")
with open(profiles_output_path, "w", encoding="utf-8") as f:
    json.dump(profiles_for_export, f, indent=2, ensure_ascii=False)
print(f"✅ Profiles exported to: {profiles_output_path}")


# === 3. Export Teams as CSV (for quick viewing) ===
csv_rows = []
for team in final_teams:
    for member in team["members"]:
        csv_rows.append({
            "Project": team["project_name"],
            "Student_ID": member["student_id"],
            "Name": member["name"],
            "Role": member["role"],
            "Match_Score": member["match_score"],
            "Experience_Score": member["experience_score"],
            "Reason": member["reason"],
            "Team_Balance": team["balance_score"]["overall_balance"],
        })

df_teams = pd.DataFrame(csv_rows)
csv_output_path = os.path.join(OUTPUT_DIR, "team_assignments.csv")
df_teams.to_csv(csv_output_path, index=False)
print(f"✅ CSV exported to: {csv_output_path}")


# === 4. Export Student Profiles as CSV ===
profile_rows = []
for p in student_profiles:
    row = {
        "Student_ID": p["student_id"],
        "Name": p["name"],
        "Primary_Role": ROLE_DISPLAY_NAMES.get(p["primary_role"], p["primary_role"]),
        "Experience_Score": p["experience_score"],
        "Skill_Diversity": p["skill_diversity"],
    }
    # Add individual role scores
    for role in ALL_ROLES:
        row[f"Skill_{ROLE_DISPLAY_NAMES.get(role, role)}"] = p["skills"].get(role, 0)
    profile_rows.append(row)

df_profiles = pd.DataFrame(profile_rows)
profiles_csv_path = os.path.join(OUTPUT_DIR, "student_profiles.csv")
df_profiles.to_csv(profiles_csv_path, index=False)
print(f"✅ Profiles CSV exported to: {profiles_csv_path}")

print(f"\n📁 All outputs saved to: {OUTPUT_DIR}/")
print(f"   - final_teams.json     (Teams with explanations)")
print(f"   - student_profiles.json (All student profiles)")
print(f"   - team_assignments.csv  (Team assignments table)")
print(f"   - student_profiles.csv  (Student profiles table)")

---
## 📊 Step 12: Visualization & Summary

Generate visual summaries of the team formation results for your demo/viva.

In [ ]:
# ============================================================
# VISUALIZATION — Summary charts and statistics
# ============================================================
# Note: On Google Colab, matplotlib is pre-installed.
# Uncomment the install line below if running locally:
# !pip install matplotlib -q

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Color palette for roles
ROLE_COLORS = {
    "frontend": "#61DAFB",
    "backend": "#68A063",
    "fullstack": "#FF6B6B",
    "ml": "#FF9F43",
    "data": "#A29BFE",
    "uiux": "#FD79A8",
    "devops": "#00CEC9",
}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("SkillSyncAI — Team Formation Analysis", fontsize=16, fontweight='bold')

# --- Plot 1: Role Distribution (Pie Chart) ---
ax1 = axes[0, 0]
role_counts = Counter(p["primary_role"] for p in student_profiles)
labels = [ROLE_DISPLAY_NAMES.get(r, r) for r in role_counts.keys()]
sizes = list(role_counts.values())
colors = [ROLE_COLORS.get(r, "#999") for r in role_counts.keys()]
ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%', startangle=90)
ax1.set_title("Student Role Distribution")

# --- Plot 2: Team Balance Scores (Bar Chart) ---
ax2 = axes[0, 1]
team_names = [t["project_name"] for t in final_teams]
balance_scores = [t["balance_score"]["overall_balance"] for t in final_teams]
bars = ax2.barh(team_names, balance_scores, color='#6C5CE7')
ax2.set_xlim(0, 1)
ax2.set_xlabel("Balance Score")
ax2.set_title("Team Balance Scores")
for bar, score in zip(bars, balance_scores):
    ax2.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2,
             f'{score:.2f}', va='center', fontsize=10)

# --- Plot 3: Experience Score Distribution (Histogram) ---
ax3 = axes[1, 0]
exp_scores = [p["experience_score"] for p in student_profiles]
ax3.hist(exp_scores, bins=10, color='#FDCB6E', edgecolor='#E17055', alpha=0.8)
ax3.set_xlabel("Experience Score (0-5)")
ax3.set_ylabel("Number of Students")
ax3.set_title("Experience Score Distribution")
ax3.axvline(sum(exp_scores)/len(exp_scores) if exp_scores else 0,
            color='red', linestyle='--', label=f'Mean: {sum(exp_scores)/max(len(exp_scores),1):.1f}')
ax3.legend()

# --- Plot 4: Skill Diversity Distribution (Box Plot) ---
ax4 = axes[1, 1]
if final_teams:
    team_data = []
    team_labels = []
    for team in final_teams:
        scores = [m["match_score"] for m in team["members"] if "match_score" in m]
        if scores:
            team_data.append(scores)
            team_labels.append(team["project_name"][:15])

    if team_data:
        bp = ax4.boxplot(team_data, labels=team_labels, patch_artist=True)
        for patch in bp['boxes']:
            patch.set_facecolor('#74B9FF')
ax4.set_ylabel("Match Score")
ax4.set_title("Match Score Distribution per Team")
ax4.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "analysis_charts.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n📊 Charts saved to: {OUTPUT_DIR}/analysis_charts.png")

In [ ]:
# ============================================================
# 📋 FINAL SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("📋 SKILLSYNCAI — EXECUTION SUMMARY")
print("=" * 70)
print(f"\n   📄 Resumes Processed:    {len(raw_resumes)}")
print(f"   👤 Profiles Built:       {len(student_profiles)}")
print(f"   🎯 Projects Configured:  {len(parsed_projects)}")
print(f"   🏆 Teams Formed:         {len(final_teams)}")

total_assigned = sum(len(t['members']) for t in final_teams)
total_unassigned = len(student_profiles) - total_assigned
print(f"   ✅ Students Assigned:    {total_assigned}")
print(f"   ❌ Students Unassigned:  {total_unassigned}")

if final_teams:
    avg_balance = sum(t['balance_score']['overall_balance'] for t in final_teams) / len(final_teams)
    avg_coverage = sum(t['balance_score']['role_coverage'] for t in final_teams) / len(final_teams)
    print(f"\n   ⚖️ Avg Team Balance:    {avg_balance:.3f}")
    print(f"   📌 Avg Role Coverage:   {avg_coverage:.0%}")

print(f"\n   📁 Output Directory:    {os.path.abspath(OUTPUT_DIR)}")
print("\n" + "=" * 70)
print("✅ SkillSyncAI pipeline complete!")
print("=" * 70)

---
## 🎓 Academic Notes (For Viva/Review)

### Techniques Demonstrated
| Technique | Implementation |
|-----------|---------------|
| **NLP** | Rule-based keyword extraction + spaCy NER for name detection |
| **Feature Engineering** | Skill scoring, experience heuristics, diversity metrics |
| **Role-Based Matching** | Weighted formula with configurable parameters |
| **Constraint-Based Optimization** | Greedy + round-robin with fairness balancing |
| **Explainable AI (XAI)** | Human-readable reasons for every team assignment |

### Matching Formula
$$\text{match\_score} = 0.6 \times \text{role\_skill} + 0.3 \times \text{experience} + 0.1 \times \text{diversity} + \text{bonus}$$

### Future Scope
- Semantic embeddings (Word2Vec / BERT) for better skill matching
- GitHub / LinkedIn profile integration
- Optimization algorithms (Hungarian Algorithm, ILP)
- Feedback loop from team performance

---
*SkillSyncAI — Built for academic evaluation*